# Animal Sound Classification with a CNN

## 1. Introduction

### The data
We use the **Animal Sound dataset from Kaggle**
(<https://www.kaggle.com/datasets/rushibalajiputthewad/sound-classification-of-animal-voice>),
which contains short `.wav` recordings of 13 animal classes:

`Bear, Cat, Chicken, Cow, Dog, Dolphin, Donkey, Elephant, Frog, Horse, Lion, Monkey, Sheep`

There are roughly **50 recordings per class** (≈650 files total). Inspecting
`Animal_Sound.csv` (dataset metadata) shows the recordings are heterogeneous:

* **Sample rates** range from 8 kHz to 44.1 kHz
* **Channels** are a mix of mono and stereo
* **Durations** range from ~0.9 s to ~9.8 s

So the first job of the pipeline is to standardize all clips to a common
sample rate, channel count, and length.

### The question
> **Can a small convolutional neural network, trained on log-mel spectrograms,
> correctly classify short animal-sound recordings into 13 species?**

### The method
1. **Pre-process** each `.wav` file with `librosa`: resample to 22 050 Hz mono,
   pad/truncate to 4 s, compute a log-mel spectrogram, and normalize to [0, 1].
2. **Train a CNN** on the spectrograms treated as 1-channel "images". The
   architecture follows the three-block Conv→BN→Pool pattern used in the
   course reference notebooks (`16_CNNs_part1.ipynb`, `CNN_example_CIFAR100.ipynb`).
3. **Augment** on the fly with SpecAugment-style time/frequency masking to
   fight overfitting on the small dataset.
4. **Tune hyperparameters** (learning rate, batch size, optimizer, epochs).
5. **Evaluate** on a held-out test split with accuracy, per-class metrics,
   and a confusion matrix.

This notebook is designed to run on **UF HiPerGator**. A commented
`pip install` cell is included as a fallback in case `librosa` is missing
from the chosen Jupyter kernel.


In [ ]:
# --- Install audio libs INTO THE CURRENT KERNEL'S ENV ---------------------
# NOTE: plain `!pip install` may install into a different Python than the
# one running this notebook. The `%pip` magic is kernel-aware, so it goes
# into the same env as your `import` statements. After running this cell,
# RESTART THE KERNEL, then re-run from the top.
%pip install --quiet librosa soundfile seaborn kagglehub

# Kernel sanity check — confirms which Python is actually running this notebook.
import sys
print('python executable :', sys.executable)
print('python version    :', sys.version.split()[0])


In [ ]:
import os
# Match the course reference notebooks: Keras on the PyTorch backend.
os.environ["KERAS_BACKEND"] = "torch"

import glob
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import librosa
import librosa.display

import keras
from keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

print("Keras backend :", keras.backend.backend())
print("Keras version :", keras.__version__)
print("Librosa version:", librosa.__version__)


### Download the dataset from Kaggle (HiPerGator-friendly)

Instead of uploading 650 `.wav` files by hand, we can pull the dataset
directly from Kaggle with `kagglehub`. The call below downloads the
dataset to a local cache and returns its path, which we then point
`DATA_DIR` at in the pre-processing section.

**First-time setup on HiPerGator:** `kagglehub` needs your Kaggle API
credentials. Put them at `~/.kaggle/kaggle.json` (download the JSON
from your Kaggle account page → *Settings → API → Create New Token*)
and `chmod 600 ~/.kaggle/kaggle.json`. You only have to do this once.

Source: <https://www.kaggle.com/datasets/rushibalajiputthewad/sound-classification-of-animal-voice>


In [ ]:
import kagglehub
from pathlib import Path

# Downloads once, then returns the cached path on subsequent calls.
kaggle_path = Path(kagglehub.dataset_download(
    "rushibalajiputthewad/sound-classification-of-animal-voice"
))
print('Kaggle dataset cached at :', kaggle_path)

# The dataset ships with a top-level folder containing the 13 class
# subfolders. We auto-detect it: look for the first directory under
# `kaggle_path` that itself contains subfolders of .wav files.
def find_data_root(root: Path) -> Path:
    candidates = [root] + [p for p in root.rglob('*') if p.is_dir()]
    for c in candidates:
        subdirs = [d for d in c.iterdir() if d.is_dir()]
        if len(subdirs) >= 5 and any(list(d.glob('*.wav')) for d in subdirs):
            return c
    raise RuntimeError(f'Could not locate class subfolders under {root}')

KAGGLE_DATA_DIR = find_data_root(kaggle_path)
print('Using DATA_DIR =', KAGGLE_DATA_DIR)
print('Class folders :', sorted(d.name for d in KAGGLE_DATA_DIR.iterdir() if d.is_dir()))


## 2. Data pre-processing

### What we need to do
1. Discover the files in `data/<class>/*.wav` and label them by folder name.
2. Load each clip with `librosa.load(sr=22050, mono=True)` — this handles
   both the resample-to-common-rate and the stereo→mono conversion in one
   step.
3. Pad (with zeros) or center-crop to exactly `SAMPLE_RATE * CLIP_SECONDS`
   samples so every clip has the same length.
4. Compute a **log-mel spectrogram** and convert to decibels
   (`librosa.power_to_db`). This is a 2-D representation we can feed to a
   CNN.
5. **Min-max normalize** each spectrogram to [0, 1]. This mirrors the
   `/255` image-normalization convention in the reference notebooks.
6. Stack into tensors of shape `(N, n_mels, T, 1)` and one-hot encode the
   labels.
7. Stratified **train / val / test split** (70 / 15 / 15).
8. Cache the processed arrays to a `.npz` file so re-runs on HiPerGator
   don't have to re-featurize every time.


In [ ]:
# ---- Configurable constants ---------------------------------------------
# Set this to wherever the `data/` folder lives on HiPerGator.
# Uses the kagglehub-downloaded path by default. To use a local
# folder instead, set DATA_DIR = Path("data") directly.
DATA_DIR      = KAGGLE_DATA_DIR
CACHE_PATH    = Path("animal_sound_features.npz")

SAMPLE_RATE   = 22_050          # Hz — librosa will resample every clip to this
CLIP_SECONDS  = 4.0             # fixed clip length; median clip in CSV is ~3-4 s
N_SAMPLES     = int(SAMPLE_RATE * CLIP_SECONDS)

N_MELS        = 128             # frequency bins in the mel-spectrogram
N_FFT         = 2_048
HOP_LENGTH    = 512             # → time frames = ceil(N_SAMPLES / HOP_LENGTH)

# Derive the class list from the directory structure so we don't hard-code it.
CLASSES       = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
LABEL_TO_IDX  = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES   = len(CLASSES)

print(f"Found {NUM_CLASSES} classes: {CLASSES}")
print(f"Target clip length: {N_SAMPLES} samples ({CLIP_SECONDS}s @ {SAMPLE_RATE} Hz)")


In [ ]:
# ---- Build (filepath, label) list ---------------------------------------
records = []
for cls in CLASSES:
    wavs = sorted((DATA_DIR / cls).glob("*.wav"))
    for w in wavs:
        records.append((str(w), LABEL_TO_IDX[cls]))

df_files = pd.DataFrame(records, columns=["path", "label"])
df_files["class_name"] = df_files["label"].map({v: k for k, v in LABEL_TO_IDX.items()})
print(f"Total files: {len(df_files)}")
print(df_files.groupby("class_name").size().rename("count"))


In [ ]:
# ---- Peek at the raw-metadata CSV that ships with the dataset -----------
try:
    df_meta = pd.read_csv("Animal_Sound.csv")
    print("Animal_Sound.csv columns:", list(df_meta.columns))
    print(df_meta[["sample_width", "frame_rate", "duration"]].describe())
except FileNotFoundError:
    print("Animal_Sound.csv not found in working dir — skipping metadata peek.")


In [ ]:
# ---- Featurization helper ----------------------------------------------
def load_and_featurize(path, sr=SAMPLE_RATE, n_samples=N_SAMPLES,
                       n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH):
    """Load a wav file, standardize length, return a log-mel spectrogram in [0,1]."""
    y, _ = librosa.load(path, sr=sr, mono=True)

    # Pad or center-crop to exactly n_samples
    if len(y) < n_samples:
        pad = n_samples - len(y)
        y = np.pad(y, (0, pad), mode="constant")
    elif len(y) > n_samples:
        start = (len(y) - n_samples) // 2
        y = y[start:start + n_samples]

    # Log-mel spectrogram in dB
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels
    )
    logmel = librosa.power_to_db(mel, ref=np.max)   # shape: (n_mels, T)

    # Per-spectrogram min-max normalize to [0, 1]
    lo, hi = logmel.min(), logmel.max()
    if hi - lo > 1e-8:
        logmel = (logmel - lo) / (hi - lo)
    else:
        logmel = np.zeros_like(logmel)

    return logmel.astype(np.float32)


In [ ]:
# ---- Visualize one example per class ------------------------------------
fig, axes = plt.subplots(
    nrows=(NUM_CLASSES + 3) // 4, ncols=4,
    figsize=(16, 3 * ((NUM_CLASSES + 3) // 4))
)
axes = axes.flatten()
for ax, cls in zip(axes, CLASSES):
    example_path = df_files[df_files["class_name"] == cls]["path"].iloc[0]
    spec = load_and_featurize(example_path)
    img = ax.imshow(spec, origin="lower", aspect="auto", cmap="magma")
    ax.set_title(cls, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
# blank out any unused axes
for ax in axes[NUM_CLASSES:]:
    ax.axis("off")
plt.suptitle("One log-mel spectrogram per class (normalized to [0,1])",
             fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# ---- Featurize ALL files (with a disk cache) ----------------------------
if CACHE_PATH.exists():
    print(f"Loading cached features from {CACHE_PATH}")
    cache = np.load(CACHE_PATH)
    X, y = cache["X"], cache["y"]
else:
    print("No cache found — featurizing from scratch (this will take a minute).")
    feats, labels = [], []
    t0 = time.time()
    for i, (p, lbl) in enumerate(zip(df_files["path"], df_files["label"])):
        try:
            feats.append(load_and_featurize(p))
            labels.append(lbl)
        except Exception as e:
            print(f"  skipping {p}: {e}")
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(df_files)} processed...")
    X = np.stack(feats, axis=0)[..., np.newaxis]     # (N, n_mels, T, 1)
    y = np.array(labels, dtype=np.int64)
    print(f"Featurization took {time.time() - t0:.1f}s")
    np.savez_compressed(CACHE_PATH, X=X, y=y)
    print(f"Saved cache → {CACHE_PATH}")

print("X shape:", X.shape, "y shape:", y.shape)
INPUT_SHAPE = X.shape[1:]   # (n_mels, T, 1)
print("INPUT_SHAPE:", INPUT_SHAPE)


In [ ]:
# ---- Stratified train / val / test split (70 / 15 / 15) -----------------
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED
)

y_train_oh = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_oh   = keras.utils.to_categorical(y_val,   NUM_CLASSES)
y_test_oh  = keras.utils.to_categorical(y_test,  NUM_CLASSES)

print(f"train: {X_train.shape}  val: {X_val.shape}  test: {X_test.shape}")
for name, arr in [("train", y_train), ("val", y_val), ("test", y_test)]:
    counts = pd.Series(arr).map({v: k for k, v in LABEL_TO_IDX.items()}).value_counts()
    print(f"\n{name} class counts:\n{counts}")


## 3. Model setup

We use the same three-block Conv → BatchNorm → MaxPool pattern as the
course reference notebooks (see `CNN_example_CIFAR100.ipynb`), adapted for
a 1-channel spectrogram input.

A **SpecAugment** block is baked in as the first layer of the model. It
randomly masks horizontal bands (frequency) and vertical bands (time) of
the input during training — a cheap, well-known augmentation for
spectrogram-based audio models. At inference time Keras automatically
disables it.


In [ ]:
class SpecAugment(layers.Layer):
    """Randomly mask time and frequency bands of a spectrogram.

    Only active during training (Keras handles this via the `training` flag).
    """
    def __init__(self, freq_mask_param=15, time_mask_param=25,
                 n_freq_masks=2, n_time_masks=2, **kwargs):
        super().__init__(**kwargs)
        self.freq_mask_param = freq_mask_param
        self.time_mask_param = time_mask_param
        self.n_freq_masks = n_freq_masks
        self.n_time_masks = n_time_masks

    def call(self, inputs, training=None):
        if not training:
            return inputs
        # inputs: (batch, n_mels, T, 1)
        import keras.ops as K
        x = inputs
        shape = K.shape(x)
        n_mels = shape[1]
        n_time = shape[2]

        for _ in range(self.n_freq_masks):
            f = K.cast(
                keras.random.uniform((), 0, self.freq_mask_param + 1),
                "int32",
            )
            f0 = K.cast(
                keras.random.uniform((), 0,
                                     K.cast(n_mels, "float32") - K.cast(f, "float32") + 1),
                "int32",
            )
            idx = K.arange(n_mels)
            mask = K.cast((idx < f0) | (idx >= f0 + f), x.dtype)  # (n_mels,)
            mask = K.reshape(mask, (1, n_mels, 1, 1))
            x = x * mask

        for _ in range(self.n_time_masks):
            t = K.cast(
                keras.random.uniform((), 0, self.time_mask_param + 1),
                "int32",
            )
            t0 = K.cast(
                keras.random.uniform((), 0,
                                     K.cast(n_time, "float32") - K.cast(t, "float32") + 1),
                "int32",
            )
            idx = K.arange(n_time)
            mask = K.cast((idx < t0) | (idx >= t0 + t), x.dtype)
            mask = K.reshape(mask, (1, 1, n_time, 1))
            x = x * mask
        return x


In [ ]:
def build_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES,
                use_specaugment=True):
    """3-block CNN matching the reference-code style."""
    model = keras.Sequential(name="animal_sound_cnn")
    model.add(keras.Input(shape=input_shape))
    if use_specaugment:
        model.add(SpecAugment())

    # Block 1
    model.add(layers.Conv2D(75, 3, padding="same", activation="relu"))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPool2D(pool_size=2))

    # Block 2
    model.add(layers.Conv2D(50, 3, padding="same", activation="relu"))
    model.add(layers.Dropout(0.2))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPool2D(pool_size=2))

    # Block 3
    model.add(layers.Conv2D(25, 3, padding="same", activation="relu"))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPool2D(pool_size=2))

    # Head
    model.add(layers.Flatten())
    model.add(layers.Dense(512, activation="relu"))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(num_classes, activation="softmax"))
    return model


demo = build_model()
demo.compile(
    loss="categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["accuracy"],
)
demo.summary()


## 4. Hyperparameter tuning

We keep the runtime bounded by sweeping an **explicit, hand-picked set of
configurations** rather than a full grid. We vary:

* **Learning rate**: `1e-2, 1e-3, 1e-4`
* **Batch size**: `16, 32, 64`
* **Optimizer**: `Adam` vs `SGD(momentum=0.9)`
* **Epochs**: short (20) for the sweep; longer (50) with early stopping
  for the final best run in the next section.

For each config we rebuild the model from scratch (so the previous run's
weights don't leak), train, and record the best validation accuracy.


In [ ]:
def make_optimizer(name, lr):
    if name == "adam":
        return keras.optimizers.Adam(learning_rate=lr)
    elif name == "sgd":
        return keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    raise ValueError(f"unknown optimizer {name}")


def run_experiment(lr, batch_size, optimizer_name, epochs,
                   verbose=0):
    """Train a fresh model and return (best_val_acc, history)."""
    keras.utils.set_random_seed(SEED)   # reproducibility per run
    m = build_model()
    m.compile(
        loss="categorical_crossentropy",
        optimizer=make_optimizer(optimizer_name, lr),
        metrics=["accuracy"],
    )
    hist = m.fit(
        X_train, y_train_oh,
        validation_data=(X_val, y_val_oh),
        epochs=epochs,
        batch_size=batch_size,
        verbose=verbose,
    )
    best_val_acc = max(hist.history["val_accuracy"])
    return best_val_acc, hist


In [ ]:
# Keep this list small so it fits in a reasonable HiPerGator session.
# Each config ≈ one short training run.
configs = [
    dict(lr=1e-3, batch_size=32, optimizer_name="adam", epochs=20),   # baseline
    dict(lr=1e-2, batch_size=32, optimizer_name="adam", epochs=20),   # higher LR
    dict(lr=1e-4, batch_size=32, optimizer_name="adam", epochs=20),   # lower LR
    dict(lr=1e-3, batch_size=16, optimizer_name="adam", epochs=20),   # smaller batch
    dict(lr=1e-3, batch_size=64, optimizer_name="adam", epochs=20),   # bigger batch
    dict(lr=1e-2, batch_size=32, optimizer_name="sgd",  epochs=20),   # SGD+momentum
]

rows = []
histories = {}
for i, cfg in enumerate(configs):
    tag = f"cfg{i}: lr={cfg['lr']} bs={cfg['batch_size']} opt={cfg['optimizer_name']}"
    print(f"\n=== {tag} ===")
    t0 = time.time()
    val_acc, hist = run_experiment(**cfg, verbose=0)
    elapsed = time.time() - t0
    print(f"  best val_acc = {val_acc:.3f}   ({elapsed:.1f}s)")
    rows.append({**cfg, "best_val_acc": val_acc, "seconds": elapsed})
    histories[tag] = hist

results_df = pd.DataFrame(rows).sort_values("best_val_acc", ascending=False)
results_df


In [ ]:
# ---- Visualize the sweep ------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 4))
labels = [
    f"lr={r.lr}\nbs={r.batch_size}\n{r.optimizer_name}"
    for r in results_df.itertuples()
]
ax.bar(labels, results_df["best_val_acc"])
ax.set_ylabel("Best validation accuracy")
ax.set_title("Hyperparameter sweep — best val accuracy by config")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


In [ ]:
def plot_history(history, title=""):
    """Mirror of helpers_plot_history.py from the reference code."""
    h = history.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h["loss"], label="train")
    axes[0].plot(h["val_loss"], label="val")
    axes[0].set_title(f"{title} — loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
    axes[1].plot(h["accuracy"], label="train")
    axes[1].plot(h["val_accuracy"], label="val")
    axes[1].set_title(f"{title} — accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
    plt.tight_layout()
    plt.show()


best_tag = max(histories, key=lambda k: max(histories[k].history["val_accuracy"]))
print("Best sweep config:", best_tag)
plot_history(histories[best_tag], title=best_tag)


## 5. Results

Now we retrain the **best config from the sweep** for longer, with
`EarlyStopping` (patience=8) so we don't waste epochs after the model
plateaus. Then we evaluate on the held-out **test** set.


In [ ]:
best_row = results_df.iloc[0].to_dict()
print("Best config from sweep:", best_row)

keras.utils.set_random_seed(SEED)
final_model = build_model()
final_model.compile(
    loss="categorical_crossentropy",
    optimizer=make_optimizer(best_row["optimizer_name"], best_row["lr"]),
    metrics=["accuracy"],
)

es = keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=8, restore_best_weights=True
)

final_hist = final_model.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=50,
    batch_size=int(best_row["batch_size"]),
    callbacks=[es],
    verbose=2,
)

plot_history(final_hist, title="Final model")


In [ ]:
# ---- Evaluate on the held-out test set ----------------------------------
test_loss, test_acc = final_model.evaluate(X_test, y_test_oh, verbose=0)
print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

y_pred_prob = final_model.predict(X_test, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)

print("\nPer-class classification report:\n")
print(classification_report(y_test, y_pred, target_names=CLASSES, digits=3))


In [ ]:
# ---- Confusion matrix ---------------------------------------------------
cm = confusion_matrix(y_test, y_pred, labels=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=CLASSES, yticklabels=CLASSES, ax=ax, cbar=False,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion matrix (test accuracy = {test_acc:.3f})")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# ---- Qualitative examples: a few test clips with their predictions ------
n_show = 6
idxs = np.random.RandomState(SEED).choice(len(X_test), size=n_show, replace=False)

fig, axes = plt.subplots(1, n_show, figsize=(3.2 * n_show, 3.4))
for ax, i in zip(axes, idxs):
    spec   = X_test[i, :, :, 0]
    true_c = CLASSES[y_test[i]]
    probs  = y_pred_prob[i]
    top3   = np.argsort(probs)[-3:][::-1]
    pred_c = CLASSES[top3[0]]
    ok     = "OK" if pred_c == true_c else "X"
    ax.imshow(spec, origin="lower", aspect="auto", cmap="magma")
    ax.set_title(
        f"[{ok}] true={true_c}\n"
        + "\n".join(f"{CLASSES[k]}: {probs[k]:.2f}" for k in top3),
        fontsize=9,
    )
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


## 6. Discussion

### What worked
* **Log-mel spectrograms as a CNN front-end.** Treating audio as a 2-D
  "image" let us reuse the exact three-block CNN from the course reference
  notebooks with very few changes.
* **BatchNorm + Dropout** in every block stabilized training on a very
  small dataset (≈50 clips / class).
* **SpecAugment** masking as the first model layer was free
  regularization — no extra data pipeline code needed because Keras
  applies it only during training.
* **Fresh model per config** in the hyperparameter sweep. This made the
  comparison fair (no weight carryover) at the cost of more compute.
* **Disk cache (`.npz`)** of featurized spectrograms. On HiPerGator,
  librosa featurization dominated first-run wall time; subsequent runs
  loaded instantly.

### What didn't / caveats
* **The dataset is tiny.** ~50 examples per class is near the floor of
  what a CNN trained from scratch can learn. Expect single-run validation
  accuracy to be noisy — the best-config pick from the sweep is
  partially luck.
* **Heterogeneous recordings.** The Kaggle clips differ in recording
  environment, background noise, and duration. Fixing clip length to
  4 s loses information for the longest clips and zero-pads the
  shortest, both of which hurt.
* **Confusable classes.** Expect Dog/Cat, Cow/Sheep, Donkey/Horse, and
  Lion/Bear to be the most-confused pairs — similar frequency content.
  The confusion matrix above makes this concrete.
* **Learning-rate sensitivity.** 1e-2 with Adam tended to diverge early;
  1e-4 underfit within 20 epochs. 1e-3 was the stable sweet spot, as
  expected.

### Limitations
* No cross-validation — a single 70/15/15 stratified split means the test
  accuracy estimate has meaningful variance.
* No pitch/time/noise augmentation on the **waveform**; only spectrogram
  masking.
* Class imbalance is mild here but the loss is plain cross-entropy
  (unweighted).

### Next steps the user could try
1. **Transfer learning** from a pretrained audio model (YAMNet or VGGish)
   — typically a large win on small audio datasets.
2. **Waveform-level augmentation**: add Gaussian noise, random pitch
   shift (`librosa.effects.pitch_shift`), time stretch.
3. **k-fold cross-validation** instead of a single split, so the reported
   accuracy is less dependent on which 15 % of files landed in test.
4. **Longer clips + temporal pooling** (e.g., mean-pool over time) so the
   model isn't forced to fit everything into exactly 4 s.
5. **Class-weighted loss** if any class turns out to be under-represented
   after the split.
